# 03 Semantic Partition And Evaluation

Assignment step 6: partition global `V` into measures `M`, dimension names `N`, dimension values `A`, units `U`, plus `other_ambiguous` for unsafe cases.


In [1]:
from pathlib import Path
import csv
import json

import pyarrow.parquet as pq

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent


def read_json(relative_path: str):
    path = ROOT / relative_path
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {"missing": str(path)}


def csv_rows(relative_path: str, limit: int | None = None):
    csv.field_size_limit(2_147_483_647)
    path = ROOT / relative_path
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        rows = list(csv.DictReader(file))
    return rows if limit is None else rows[:limit]


def csv_count(relative_path: str) -> int | None:
    path = ROOT / relative_path
    if not path.exists():
        return None
    return len(csv_rows(relative_path))


def parquet_count(relative_path: str) -> int | None:
    path = ROOT / relative_path
    return pq.read_table(path).num_rows if path.exists() else None


## Required Output Counts


In [2]:
{
    "M measures.csv": csv_count("outputs/measures.csv"),
    "N dimension_names.csv": csv_count("outputs/dimension_names.csv"),
    "A dimension_values.csv": csv_count("outputs/dimension_values.csv"),
    "U units.csv": csv_count("outputs/units.csv"),
    "other_ambiguous.csv": csv_count("outputs/other_ambiguous.csv"),
}


{'M measures.csv': 632,
 'N dimension_names.csv': 365,
 'A dimension_values.csv': 548,
 'U units.csv': 214,
 'other_ambiguous.csv': 7812}

## Quality Evaluation


In [3]:
metrics = read_json("report/classification_metrics.json")
{
    "variant": metrics.get("variant"),
    "gold_status": metrics.get("gold_status"),
    "agreement": metrics.get("agreement"),
    "overall": metrics.get("metrics", {}),
    "validation": metrics.get("validation_metrics", {}),
    "final_test": metrics.get("final_test_metrics", {}),
}


{'variant': 'local-hybrid',
 'gold_status': 'available',
 'agreement': {'cohen_kappa': 1.0,
  'disagreements': [],
  'raw_agreement': 1.0,
  'reviewed_count': 50,
  'status': 'available'},
 'overall': {'accuracy': 0.948,
  'confusion_matrix': {'dimension_name': {'dimension_name': 86,
    'dimension_value': 0,
    'measure': 0,
    'other_ambiguous': 0,
    'unit': 0},
   'dimension_value': {'dimension_name': 0,
    'dimension_value': 26,
    'measure': 0,
    'other_ambiguous': 0,
    'unit': 0},
   'measure': {'dimension_name': 0,
    'dimension_value': 0,
    'measure': 23,
    'other_ambiguous': 26,
    'unit': 0},
   'other_ambiguous': {'dimension_name': 0,
    'dimension_value': 0,
    'measure': 0,
    'other_ambiguous': 320,
    'unit': 0},
   'unit': {'dimension_name': 0,
    'dimension_value': 0,
    'measure': 0,
    'other_ambiguous': 0,
    'unit': 19}},
  'gold_count': 500,
  'macro_f1': 0.9199699699699699,
  'missing_predictions': [],
  'per_class': {'dimension_name': {'f

## Example Classified Terms


In [4]:
{
    "measures": csv_rows("outputs/measures.csv", 3),
    "dimension_names": csv_rows("outputs/dimension_names.csv", 3),
    "dimension_values": csv_rows("outputs/dimension_values.csv", 3),
    "units": csv_rows("outputs/units.csv", 3),
    "other_ambiguous": csv_rows("outputs/other_ambiguous.csv", 3),
}


{'measures': [{'term_id': 'term_20d6b4a1cc5307075745',
   'canonical_term': 'Accidents in waterway transport',
   'category': 'measure',
   'confidence': '0.88',
   'variant': 'local-hybrid',
   'protected': 'True',
   'evidence': 'title evidence with measure lexical cue',
   'occurrence_ids_json': '["occ_5f8eb80b7e42c67847f1"]',
   'run_id': 'run_75b90575bb9e48ce7918'},
  {'term_id': 'term_2c16cafab319d30eed0d',
   'canonical_term': 'Accidents involving the transport of dangerous goods',
   'category': 'measure',
   'confidence': '0.88',
   'variant': 'local-hybrid',
   'protected': 'True',
   'evidence': 'title evidence with measure lexical cue',
   'occurrence_ids_json': '["occ_84ff46631dc99865b39e"]',
   'run_id': 'run_75b90575bb9e48ce7918'},
  {'term_id': 'term_0f8f9cf7bf3053d8d92c',
   'canonical_term': 'Accidents involving the transport of dangerous goods - annual data',
   'category': 'measure',
   'confidence': '0.88',
   'variant': 'local-hybrid',
   'protected': 'True',
   '